7.0.1 Drive i putanje

Montira se Drive i postavljaju se putanje ka podacima i Runs folderu. Ovaj notebook upisuje sve artefakte u Runs/final_YYYYMMDD_HHMMSS.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, math
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Diplomski")
DATA = ROOT / "Data"
CURATED = DATA / "curated"
CURATED_QC = DATA / "curated_qc"
CURATED_V2 = DATA / "curated_v2"
RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("%Y%m%d_%H%M%S")
OUT = RUNS / f"final_{run_id}"
OUT.mkdir(parents=True, exist_ok=True)

print("OUT:", OUT)

Mounted at /content/drive
OUT: /content/drive/MyDrive/Diplomski/Runs/final_20260319_123310


7.0.2 Registry helperi

Učitavaju se meta i splitovi iz različitih verzija (curated, curated_qc, curated_v2). Cilj je da registry bude jedinstvena “istina” gde se šta nalazi.

In [2]:
def load_meta(ds, root):
    p = root / ds / "meta.json"
    if not p.exists():
        return None
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split, root):
    p = root / ds / f"{split}.csv"
    if not p.exists():
        return None
    return pd.read_csv(p)

def infer_task(meta):
    if meta is None:
        return None
    t = meta.get("task_type")
    if t in ["image", "tabular"]:
        return t
    if "image" in json.dumps(meta).lower():
        return "image"
    return "tabular"

def get_basic_stats(ds, root):
    meta = load_meta(ds, root)
    if meta is None:
        return None
    df_tr = load_split(ds, "train", root)
    df_va = load_split(ds, "val", root)
    df_te = load_split(ds, "test", root)
    if df_tr is None:
        return None

    task = infer_task(meta)
    if task == "tabular":
        ycol = meta.get("label_col", None)
        if ycol is None or ycol not in df_tr.columns:
            ycol = df_tr.columns[-1]
        n_classes = int(df_tr[ycol].astype(str).nunique())
    else:
        ycol = meta.get("label_col", "label")
        if ycol not in df_tr.columns:
            ycol = df_tr.columns[-1]
        n_classes = int(df_tr[ycol].astype(str).nunique())

    return {
        "task": task,
        "n_train": int(len(df_tr)),
        "n_val": int(len(df_va)) if df_va is not None else None,
        "n_test": int(len(df_te)) if df_te is not None else None,
        "n_classes": n_classes,
    }

7.1 Dataset registry (CSV)

Ovaj korak pravi tabelu svih datasetova i dostupnih verzija splitova. Fajl Data/dataset_registry.csv je zgodan i za Word i za “kriterijum završetka”.

In [3]:
DATASETS = ["thyroid_recurrence", "lc25000", "sipakmed", "rm1000_lung_history"]
VERSIONS = [
    ("curated", CURATED),
    ("curated_qc", CURATED_QC),
    ("curated_v2", CURATED_V2),
]

rows = []
for ds in DATASETS:
    row = {"dataset": ds}
    for vname, vroot in VERSIONS:
        st = get_basic_stats(ds, vroot)
        row[f"{vname}_exists"] = bool(st is not None)
        if st is not None:
            row[f"{vname}_task"] = st["task"]
            row[f"{vname}_n_train"] = st["n_train"]
            row[f"{vname}_n_val"] = st["n_val"]
            row[f"{vname}_n_test"] = st["n_test"]
            row[f"{vname}_n_classes"] = st["n_classes"]
        else:
            row[f"{vname}_task"] = None
            row[f"{vname}_n_train"] = None
            row[f"{vname}_n_val"] = None
            row[f"{vname}_n_test"] = None
            row[f"{vname}_n_classes"] = None

    rows.append(row)

df_reg = pd.DataFrame(rows)
reg_path = DATA / "dataset_registry.csv"
df_reg.to_csv(reg_path, index=False)
print("Saved:", reg_path)
df_reg

Saved: /content/drive/MyDrive/Diplomski/Data/dataset_registry.csv


,dataset,curated_exists,curated_task,curated_n_train,curated_n_val,curated_n_test,curated_n_classes,curated_qc_exists,curated_qc_task,curated_qc_n_train,curated_qc_n_val,curated_qc_n_test,curated_qc_n_classes,curated_v2_exists,curated_v2_task,curated_v2_n_train,curated_v2_n_val,curated_v2_n_test,curated_v2_n_classes
0,thyroid_recurrence,True,tabular,255,54,55,2,False,None,NaN,NaN,NaN,NaN,False,None,NaN,NaN,NaN,NaN
1,lc25000,True,image,17500,3750,3750,2,True,image,16804.0,3605.0,3591.0,2.0,True,image,17518.0,3738.0,3744.0,2.0
2,sipakmed,True,image,2835,608,606,5,True,image,2750.0,593.0,577.0,5.0,False,None,NaN,NaN,NaN,NaN
3,rm1000_lung_history,True,image,10500,2250,2250,3,True,image,10130.0,2158.0,2157.0,3.0,True,image,10501.0,2253.0,2246.0,3.0


7.2 Final run za SipakMed (najbolja konfiguracija iz tuninga)

Ovde treniramo jedan “final” model sa malo ozbiljnijim brojem epoha i early stopping. Koristimo ResNet18, baseline augmentaciju, seed=42 (po tvom tuningu to je bila najbolja kombinacija).

In [4]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class ImageCsvDataset(torch.utils.data.Dataset):
    def __init__(self, df, tfm=None, label_to_idx=None, pcol="path", ycol="label"):
        self.df = df.reset_index(drop=True)
        self.tfm = tfm
        self.pcol = pcol
        self.ycol = ycol
        labels = self.df[self.ycol].astype(str).tolist()
        if label_to_idx is None:
            uniq = sorted(list(set(labels)))
            self.label_to_idx = {u:i for i,u in enumerate(uniq)}
        else:
            self.label_to_idx = label_to_idx
        self.y = [self.label_to_idx[str(x)] for x in labels]

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        p = str(self.df.iloc[i][self.pcol])
        y = self.y[i]
        im = Image.open(p).convert("RGB")
        if self.tfm: im = self.tfm(im)
        return im, torch.tensor(y, dtype=torch.long)

@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        pred = torch.argmax(model(x), dim=1)
        ys += y.cpu().numpy().tolist()
        ps += pred.cpu().numpy().tolist()
    return float(accuracy_score(ys, ps)), float(f1_score(ys, ps, average="macro")), ys, ps

def train_epoch(model, loader, opt, loss_fn):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float("nan")

def build_resnet18(num_classes):
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def save_confmat(cm, labels, out_path, title):
    fig, ax = plt.subplots(figsize=(6,6))
    disp = ConfusionMatrixDisplay(confusion_matrix=np.array(cm), display_labels=labels)
    disp.plot(ax=ax, values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

set_seed(42)

ds = "sipakmed"
df_tr = load_split(ds, "train", CURATED)
df_va = load_split(ds, "val", CURATED)
df_te = load_split(ds, "test", CURATED)

tfm_tr = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])
tfm_ev = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

ds_tr = ImageCsvDataset(df_tr, tfm=tfm_tr, label_to_idx=None)
ds_va = ImageCsvDataset(df_va, tfm=tfm_ev, label_to_idx=ds_tr.label_to_idx)
ds_te = ImageCsvDataset(df_te, tfm=tfm_ev, label_to_idx=ds_tr.label_to_idx)

dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=32, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
dl_va = torch.utils.data.DataLoader(ds_va, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
dl_te = torch.utils.data.DataLoader(ds_te, batch_size=32, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

model = build_resnet18(len(ds_tr.label_to_idx)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.CrossEntropyLoss()

best = {"val_f1": -1, "state": None, "epoch": None}
history = []

patience = 3
no_improve = 0
max_epochs = 10

for ep in range(1, max_epochs+1):
    tr_loss = train_epoch(model, dl_tr, opt, loss_fn)
    va_acc, va_f1, _, _ = eval_loader(model, dl_va)
    history.append({"epoch": ep, "train_loss": tr_loss, "val_acc": va_acc, "val_f1_macro": va_f1})
    print("sipakmed", "epoch", ep, "loss", tr_loss, "val_acc", va_acc, "val_f1", va_f1)

    if va_f1 > best["val_f1"]:
        best["val_f1"] = va_f1
        best["state"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        best["epoch"] = ep
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            break

if best["state"] is not None:
    model.load_state_dict({k: v.to(device) for k,v in best["state"].items()})

te_acc, te_f1, y_true, y_pred = eval_loader(model, dl_te)
cm = confusion_matrix(y_true, y_pred)
labels = [k for k,v in sorted(ds_tr.label_to_idx.items(), key=lambda x: x[1])]

final_dir = OUT / "sipakmed_resnet18_final"
final_dir.mkdir(parents=True, exist_ok=True)

with open(final_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset": ds,
        "model": "resnet18",
        "seed": 42,
        "best_epoch": best["epoch"],
        "val_best_f1": best["val_f1"],
        "test_acc": te_acc,
        "test_f1_macro": te_f1,
        "label_to_idx": ds_tr.label_to_idx,
        "history": history
    }, f, ensure_ascii=False, indent=2)

pd.DataFrame(history).to_csv(final_dir / "history.csv", index=False)
save_confmat(cm.tolist(), labels, final_dir / "confusion_matrix.png", "SipakMed Final ResNet18")

print("Saved:", final_dir)
print("Test:", te_acc, te_f1)

device: cuda
NVIDIA L4
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 201MB/s]


sipakmed epoch 1 loss 0.35011358642846013 val_acc 0.9342105263157895 val_f1 0.9336235118850761
sipakmed epoch 2 loss 0.18295030731163667 val_acc 0.9506578947368421 val_f1 0.9506956600119618
sipakmed epoch 3 loss 0.14360810234472993 val_acc 0.9523026315789473 val_f1 0.9533372924562507
sipakmed epoch 4 loss 0.09813889917614085 val_acc 0.9210526315789473 val_f1 0.9226435461693688
sipakmed epoch 5 loss 0.11300791586550434 val_acc 0.9375 val_f1 0.9367163411377785
sipakmed epoch 6 loss 0.0960788887852196 val_acc 0.9555921052631579 val_f1 0.9546919880748405
sipakmed epoch 7 loss 0.05445077632334125 val_acc 0.9555921052631579 val_f1 0.9562698674557073
sipakmed epoch 8 loss 0.055762353603037475 val_acc 0.9671052631578947 val_f1 0.966877768633324
sipakmed epoch 9 loss 0.05199523950832781 val_acc 0.9605263157894737 val_f1 0.9603450169379588
sipakmed epoch 10 loss 0.03788665809480243 val_acc 0.9638157894736842 val_f1 0.9640684147603549
Saved: /content/drive/MyDrive/Diplomski/Runs/final_20260319_12

7.3 Final summary (jedan CSV za Word)

Ovim dobijaš jednu tabelu “final rezultati” koju ubaciš u završno poglavlje.

In [5]:
final_rows = []

p = OUT / "sipakmed_resnet18_final" / "metrics.json"
m = json.loads(p.read_text(encoding="utf-8"))
final_rows.append({
    "dataset": m["dataset"],
    "final_model": m["model"],
    "seed": m["seed"],
    "test_acc": m["test_acc"],
    "test_f1_macro": m["test_f1_macro"],
    "note": f"best_epoch={m['best_epoch']}"
})

df_final = pd.DataFrame(final_rows)
df_final.to_csv(OUT / "final_summary.csv", index=False)
print("Saved:", OUT / "final_summary.csv")
df_final


Saved: /content/drive/MyDrive/Diplomski/Runs/final_20260319_123310/final_summary.csv


,dataset,final_model,seed,test_acc,test_f1_macro,note
0,sipakmed,resnet18,42,0.960396,0.960551,best_epoch=8
